# Agentic workflows with local LLMs
## Hands-on companion notebook — AI for Urban Health (Summer Institute 2026), Day 3

This notebook builds **agentic workflows** on a **local** model (Ollama) for an everyday public-health task: **triaging incoming reports**. You pick which kind of report to work with:

- **311 complaints** assigned to an environmental-health program
- **Food-safety / health-code violations** from restaurant inspections
- **Pedestrian-injury surveillance** records

You change one variable (`SCENARIO`) and the whole pipeline re-themes. Everything runs on your machine — free, private, and offline — which matters for complaint, inspection, and case data.

You will work through three increasingly capable patterns, then a tool-using example:

1. **One prompt** — a single structured-output call that triages one report.
2. **Multi-agent (CrewAI)** — a triage agent and a documentation agent collaborating.
3. **State machine (LangGraph)** — the same workflow as an explicit graph with a conditional branch.

It is an **optional, advanced track** — a richer companion to the basic Ollama activity. Your instructor may demo it live.

### Choosing a model (mid-2026)
The strongest *open-weight* model right now is **GLM-5.2**, but it is datacenter-class — not for a laptop. For local use, good options are **Gemma 4**, **Qwen3**, and small **Llama 4 / Phi-4**. See the sizing table below. *Model tags change — run `ollama list` and browse <https://ollama.com/library> for current tags.*

**Prerequisites**
- [Ollama](https://ollama.com) installed and running.
- A local model pulled (pick one that fits your laptop — see next cell):

```
ollama pull gemma4:26b   # demo default: MoE (~4B active), fast + capable (~18 GB)
ollama pull gemma4:12b   # lighter option (~7.6 GB)
```

> This notebook was demoed locally on an Apple Silicon MacBook (M1 Max, 64 GB) with **`gemma4:26b`**. On smaller laptops use `gemma4:12b`, `qwen3:8b`, or `qwen3:4b`.

### Which model fits your laptop?

Rough guide using 4-bit quantization (`Q4_K_M`, Ollama's default) — a model needs roughly **0.6 GB of memory per billion parameters**, plus headroom for context.

| Your laptop | Usable memory | Models that run well (Q4) |
|---|---|---|
| Entry (8 GB RAM, no/integrated GPU) | ~5 GB | Gemma 4 small, Qwen3 4B, Phi-4-mini, Llama 3.2 3B |
| Mainstream (16 GB RAM, or 8 GB GPU) | ~10 GB | Qwen3 8B, Gemma 4, Llama 3.1 8B |
| Strong (32 GB RAM, or 12–16 GB GPU) | ~20 GB | Qwen3 14B–32B, Gemma 4 (larger), gpt-oss 20B |
| Workstation (64 GB+ unified, or 24 GB GPU) | 40 GB+ | 70B-class or large MoE, heavily quantized |
| Not a laptop (cloud/server) | — | GLM-5.2, DeepSeek V4, MiniMax M3, Llama 4 Maverick |

**Mac (Apple Silicon):** unified memory; Ollama uses the GPU (Metal) automatically. Plan to use ~60–70% of total RAM for the model. *Intel Macs are CPU-only — stick to 3–4B models.*

**PC (Windows/Linux):** with an NVIDIA GPU the limit is VRAM (`nvidia-smi`): ~8 GB → up to 8B, 12 GB → up to 14B, 24 GB → up to ~32B. Without a GPU, Ollama runs on CPU+RAM (fine for 3–8B, slower). AMD works via ROCm.

Check what you have: `ollama list` (downloaded) and `ollama ps` (GPU vs CPU).

## 0. Setup

Install the packages we will use. One-time per environment.

> In **Positron** or VS Code, select a Python interpreter first; in Jupyter/Colab this just works.

In [ ]:
# Run once. Safe to re-run.
%pip install --quiet ollama crewai "langgraph>=0.2" "langchain-ollama>=0.2" pydantic

Confirm Ollama is running and your model is available.

In [ ]:
import ollama

MODEL = "gemma4:26b"   # demoed on Apple Silicon (M1 Max, 64 GB); lighter options: "gemma4:12b", "qwen3:8b"

available = [m.model for m in ollama.list().models]
print("Models available locally:", available)
assert any(MODEL in m for m in available), (
    f"{MODEL} not found. Run `ollama pull {MODEL}` in a terminal, or set MODEL to one of: {available}"
)
print(f"\nUsing model: {MODEL}")

## 1. Hello, local model

A bare-bones call to sanity-check your install and see what a framework does under the hood.

In [ ]:
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a public health department communications assistant. Be clear, calm, and never promise a specific timeline."},
        {"role": "user", "content": "Write a two-sentence acknowledgement to a resident who filed a 311 complaint about illegal dumping in a vacant lot."}
    ]
)
print(response.message.content)

## 2. Choose your scenario

Pick the kind of report you triage in your own work by setting `SCENARIO` to `"311"`, `"food"`, or `"pedestrian"`. Each scenario provides a handful of short, **fictional but realistic** free-text reports, a triage rule (what "needs action" means), and the fields to extract. Everything below adapts automatically.

In [ ]:
SCENARIOS = {
 "311": {
   "label": "311 environmental & housing complaints",
   "unit": "complaint",
   "triage": ("You triage 311 complaints for the Environmental Health program. A complaint NEEDS ACTION if it concerns "
              "illegal dumping/trash, vacant/blighted lots, rodents/pests, mold or lead hazards, or air/odor nuisances. "
              "It does NOT need action (route elsewhere) if it concerns potholes, streetlights, traffic signals, parking, "
              "or event noise. Mark URGENT if there is an immediate health hazard (active sewage, chemical odor, or pests "
              "in/near a home with an infant, a daycare, or a school)."),
   "fields": ["address_or_location", "complaint_type", "severity", "immediate_hazard", "recommended_next_step"],
   "draft": "a two-sentence acknowledgement to the resident (no specific timeline)",
   "tool_keyword": "rats",
   "records": [
     {"id": "C-101", "text": "Pile of construction debris and old tires dumped in the vacant lot on the 2400 block of N 5th St. It has been there two weeks and now there are rats everywhere. Kids walk past it to get to school."},
     {"id": "C-102", "text": "Streetlight has been out at 15th and Tasker for over a month. It is very dark and feels unsafe at night."},
     {"id": "C-103", "text": "Strong chemical solvent smell coming from the auto body shop on the 1900 block of Frankford Ave, worse in the mornings. It gives me headaches."},
     {"id": "C-104", "text": "Huge pothole on Broad St near Snyder Ave damaged my tire."},
     {"id": "C-105", "text": "Raw sewage standing in the basement of a rental on the 1200 block of W Allegheny. Landlord refuses to fix it and there is a baby living in the home."},
     {"id": "C-106", "text": "Trash cans at the corner store overflow constantly and there are flies all over the sidewalk."},
   ],
 },
 "food": {
   "label": "food-safety / health-code inspection reports",
   "unit": "inspection",
   "triage": ("You review restaurant inspection notes. An inspection NEEDS ACTION (re-inspection) if it has any CRITICAL "
              "violation: improper food temperatures (cold-holding above 41F, hot food held too long), cross-contamination "
              "(raw over ready-to-eat), no handwashing/hot water, or vermin/rodent evidence. It does NOT need action if only "
              "non-critical issues are noted (cosmetic, storage, minor maintenance). Mark URGENT if there is imminent risk "
              "of foodborne illness."),
   "fields": ["establishment", "critical_violations", "non_critical_violations", "reinspection_needed", "reason"],
   "draft": "a brief re-inspection notice to the establishment",
   "tool_keyword": "rodent",
   "records": [
     {"id": "F-201", "text": "Tony's Deli. No hot water at the handwashing sink. Raw chicken stored above ready-to-eat salads in the walk-in. One food handler with no hair restraint."},
     {"id": "F-202", "text": "Green Bowl Cafe. Box of single-use gloves stored on the floor. One stained ceiling tile. All food temperatures and sanitation in order."},
     {"id": "F-203", "text": "Riverside Catering. Cold-holding unit reading 52F (must be 41F or below). Cooked rice held at room temperature for 4+ hours. No sanitizer test strips on site."},
     {"id": "F-204", "text": "Sunrise Bakery. All temperatures in range, certificates current. One non-critical: a cracked floor tile near the oven."},
     {"id": "F-205", "text": "La Cocina. Rodent droppings observed near dry storage. Several dented cans on the shelf offered for sale."},
   ],
 },
 "pedestrian": {
   "label": "pedestrian-injury surveillance records",
   "unit": "report",
   "triage": ("You screen crash/EMS narratives for a pedestrian-injury surveillance system. A record NEEDS ACTION (is "
              "reportable) if a PEDESTRIAN was injured or killed on a public roadway in the city. It does NOT need action "
              "if no pedestrian was involved (driver-only, or a cyclist with no pedestrian) or it occurred off the public "
              "roadway (e.g., private parking lot) — flag those as unclear for human review. Mark URGENT if the injury was "
              "fatal or life-threatening (KSI)."),
   "fields": ["location_or_intersection", "date_or_time", "severity", "road_user", "contributing_factors"],
   "draft": "a one-line entry for the weekly surveillance summary",
   "tool_keyword": "Roosevelt",
   "records": [
     {"id": "P-301", "text": "62-year-old woman struck by a turning vehicle in the crosswalk at Roosevelt Blvd and Cottman Ave. Transported with serious leg and pelvic injuries. Driver remained on scene."},
     {"id": "P-302", "text": "Single-vehicle crash on I-95, driver only, ran off the road. No pedestrians involved."},
     {"id": "P-303", "text": "9-year-old on a bicycle clipped by an opening car door in the Spruce St bike lane. Minor abrasions, refused transport."},
     {"id": "P-304", "text": "Elderly male struck mid-block while crossing Roosevelt Blvd near Cottman Ave. Pronounced deceased at the scene."},
     {"id": "P-305", "text": "Pedestrian struck in the parking lot of a shopping center on Aramingo Ave. Moderate injuries; incident on private property."},
     {"id": "P-306", "text": "Hit-and-run: pedestrian struck at Broad St and Erie Ave, transported in serious condition."},
   ],
 },
}

SCENARIO = "311"          # change to "food" or "pedestrian"
S = SCENARIOS[SCENARIO]
RECORDS = S["records"]
print(f"Scenario: {S['label']}  ({len(RECORDS)} {S['unit']}s)")
print("Triage rule:", S["triage"][:90], "...")

## 3. Pattern 1 — one prompt with structured output

The simplest workflow: send one report plus the triage rule and get back a clean, machine-readable decision. We use Pydantic to fix the shape and have Ollama enforce JSON. Everything that follows just chains calls like this.

In [ ]:
from pydantic import BaseModel
from typing import Literal
import json

class Triage(BaseModel):
    decision: Literal["needs_action", "no_action", "unclear"]
    urgent: bool
    reason: str

def triage_one(text: str) -> Triage:
    """Triage a single report against the scenario rule."""
    r = ollama.chat(
        model=MODEL,
        format=Triage.model_json_schema(),
        messages=[
            {"role": "system", "content": f"You triage {S['label']}.\n{S['triage']}"},
            {"role": "user", "content": f"Report:\n{text}\n\nReturn a JSON triage decision."},
        ],
    )
    return Triage.model_validate_json(r.message.content)

# Test on the first record
d = triage_one(RECORDS[0]["text"])
print(d.model_dump_json(indent=2))

Now sweep the whole queue. This is already a useful little workflow.

In [ ]:
for rec in RECORDS:
    d = triage_one(rec["text"])
    flag = "  [URGENT]" if d.urgent else ""
    print(f"{rec['id']}: {d.decision.upper():12s}{flag}  — {d.reason}")

## 4. Pattern 2 — multi-agent with CrewAI

Two role-based agents collaborate: a **Triage** agent decides what needs action, and a **Documentation** agent extracts the key fields and drafts the response for the items that do. CrewAI passes messages between them; the model is the same local one.

In [ ]:
from crewai import Agent, Task, Crew, LLM, Process

llm = LLM(model=f"ollama/{MODEL}", base_url="http://localhost:11434", temperature=0.1)

triager = Agent(
    role="Intake Triage Specialist",
    goal=f"Decide which {S['unit']}s need action. {S['triage']} Return a list of IDs that need action, each with a one-line reason.",
    backstory="You have handled thousands of incoming reports and apply the rules consistently and conservatively.",
    llm=llm, verbose=False,
)
documenter = Agent(
    role="Case Documentation Specialist",
    goal=(f"For each report that needs action, extract these fields as a markdown table: {', '.join(S['fields'])}. "
          f"Then write {S['draft']} for each."),
    backstory="You turn messy field notes into clean, consistent records and clear communications.",
    llm=llm, verbose=False,
)

block = "\n\n".join(f"[{r['id']}]\n{r['text']}" for r in RECORDS)

t1 = Task(description=f"Triage rule:\n{S['triage']}\n\nReports:\n{block}\n\nList only the IDs that need action, one per line as: ID — reason.",
          expected_output="A bullet list of IDs that need action with one-line reasons.", agent=triager)
t2 = Task(description=f"Using the IDs the triager selected, extract the fields into a markdown table and draft the responses. Full reports:\n\n{block}",
          expected_output="A markdown table plus a short drafted response per selected report.", agent=documenter, context=[t1])

crew = Crew(agents=[triager, documenter], tasks=[t1, t2], process=Process.sequential, verbose=False)
print(crew.kickoff().raw)

## 5. Pattern 3 — state machine with LangGraph

The same workflow as an explicit graph: a `triage` node, then a **conditional edge** that only runs the `extract` node when the report needs action. This is more verbose than CrewAI but gives you tight control — human checkpoints, retries, per-step inspection.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm_lg = ChatOllama(model=MODEL, temperature=0.1, format="json")

class State(TypedDict):
    id: str
    text: str
    decision: str
    reason: str
    extracted: dict

def triage_node(state: State) -> State:
    msg = llm_lg.invoke([
        SystemMessage(content=f"{S['triage']}\nReply with JSON: {{\"decision\": \"needs_action|no_action|unclear\", \"reason\": \"...\"}}"),
        HumanMessage(content=f"Report:\n{state['text']}"),
    ])
    p = json.loads(msg.content)
    return {**state, "decision": p["decision"], "reason": p.get("reason", "")}

def extract_node(state: State) -> State:
    fields = ", ".join(S["fields"])
    msg = llm_lg.invoke([
        SystemMessage(content=f"Extract a JSON object with exactly these keys: {fields}. Use null for anything not stated."),
        HumanMessage(content=f"Report:\n{state['text']}"),
    ])
    return {**state, "extracted": json.loads(msg.content)}

def route(state: State) -> str:
    return "extract" if state["decision"] == "needs_action" else END

g = StateGraph(State)
g.add_node("triage", triage_node)
g.add_node("extract", extract_node)
g.set_entry_point("triage")
g.add_conditional_edges("triage", route, {"extract": "extract", END: END})
g.add_edge("extract", END)
app = g.compile()

for rec in RECORDS:
    out = app.invoke({"id": rec["id"], "text": rec["text"], "decision": "", "reason": "", "extracted": {}})
    print(f"\n--- {out['id']} ---")
    print(f"Decision: {out['decision']} — {out['reason']}")
    if out.get("extracted"):
        print("Extracted:", json.dumps(out["extracted"], indent=2))

## 6. Adding a tool — an agent that calls functions

Real agents call tools: a search, a database, a calculator. Here we give a small agent two tools over your queue — a keyword **search** and a **triage counter** (which reuses Pattern 1) — so it can answer "how many need action, and which mention X?" Conceptually, this is what **MCP** standardizes.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

@tool
def search_reports(keyword: str) -> str:
    """Search the current queue of reports for a keyword. Returns matching IDs and the first sentence."""
    hits = [f"{r['id']}: {r['text'].split('. ')[0]}." for r in RECORDS if keyword.lower() in r["text"].lower()]
    return "\n".join(hits) if hits else "No matches."

@tool
def triage_counts() -> str:
    """Triage every report in the queue and return counts of needs_action / no_action / unclear."""
    from collections import Counter
    c = Counter(triage_one(r["text"]).decision for r in RECORDS)
    return ", ".join(f"{k}: {v}" for k, v in c.items())

agent = create_react_agent(model=ChatOllama(model=MODEL, temperature=0.0),
                           tools=[search_reports, triage_counts])
result = agent.invoke({"messages": [(
    "user",
    f"How many reports in the queue need action, and which ones mention '{S['tool_keyword']}'? Use the tools."
)]})
print(result["messages"][-1].content)

## 7. Try it yourself

Pick **one** and modify the code above. Each takes 10–20 minutes.

1. **Switch scenarios.** Change `SCENARIO` to `"food"` or `"pedestrian"` and re-run everything. Where does the same model do better or worse? What would you fix in the triage rule?
2. **Add a router.** In the LangGraph version, add a node after `extract` that assigns each item to the right program/inspector/owner based on the extracted fields.
3. **Add a reviewer agent.** In the CrewAI pipeline, add a third agent that checks the documenter's table for items that look mis-triaged and flags them for a human.
4. **Detect clusters/repeats.** Add a tool that finds reports at the same location or address, and have the agent flag a possible cluster (e.g., a corridor or a repeat-offender establishment).
5. **Use your own data.** Replace one scenario's `records` with a few of your own **de-identified** reports and see how the pipeline holds up.

Write up: your change, one screenshot of the output, and one task you would *not* trust the model to do unattended.

## 8. How this maps to Day 3

- **The agent loop.** Each pattern is *perceive → decide → act → repeat* with more structure around it. Pattern 1 is one turn; CrewAI and LangGraph add multiple turns and branching.
- **Context engineering.** The model only knows what you put in the prompt: the triage rule, the report, and the fields. Most of the quality comes from that context, not the framework.
- **Tools / MCP.** Section 6 is a tool-calling agent; MCP is just a standard way to expose tools like these.
- **Guardrails.** Decisions are constrained to a fixed schema, extraction only runs on items that need action, and unstated fields are `null`.
- **Verification is the bottleneck.** Always re-read the triage calls and extracted fields against the source before acting — especially before anything reaches a resident, an establishment, or a surveillance report.

## 9. Notes and gotchas

- **JSON mode is not perfect.** Even with a schema, small models occasionally return malformed JSON. Wrap parsing in try/except and log failures.
- **Speed scales with calls.** A 6-record sweep is quick; 600 records is an overnight job. Plan accordingly.
- **Local does not mean unsupervised.** Hallucinations, weak math, and stale knowledge still apply — verify before acting.
- **De-identify first.** Keep names and exact addresses out of anything you would not put in an email; the scenarios here are fictional.
- **Triage rules are policy.** The quality of the whole pipeline depends on the triage rule you write — treat it like an SOP and version it.